# 1. Overview

This notebook runs the `tiers` workflow as a thin operator console. It restores passed samples and untiered passed manifests, runs the public `workflows.tiers` workflow, and publishes tiered manifests and tiers reports to Drive.

Workflow logic lives in `text_to_sign_production.workflows.tiers`; this notebook only configures, reviews, executes, and summarizes the public workflow contract.

# 2. Operator Configuration

Set the repository revision, runtime roots, requested splits, and tiers config paths for this run. These values are reviewed before runtime setup or workflow execution.

## 2.1 Repository and roots

Configure the repository checkout and runtime/Drive roots used by setup, restore, and publish operations.

In [ ]:
from pathlib import Path


REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "chore/core-layout-notebook-workflows"
PROJECT_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")

## 2.2 Workflow inputs

Configure the tiers workflow inputs that are passed into the public workflow config.

In [ ]:
TIERS_SPLITS = ("train", "val", "test")
FILTERS_CONFIG_RELATIVE_PATH = Path("configs/data/filters.yaml")
TIERS_CONFIG_RELATIVE_PATH = Path("configs/data/tiers.yaml")

## 2.3 Configuration review

Review the selected values before preparing the runtime.

In [ ]:
print(f"Repository URL: {REPO_URL}")
print(f"Repository ref: {REPO_REF}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Drive project root: {DRIVE_PROJECT_ROOT}")
print(f"Tiers splits: {TIERS_SPLITS}")
print(f"Filters config: {FILTERS_CONFIG_RELATIVE_PATH}")
print(f"Tiers config: {TIERS_CONFIG_RELATIVE_PATH}")

# 3. Runtime Setup

Prepare the Colab runtime and repository checkout. This section does not run workflow work.

## 3.1 Mount Drive

Mount Google Drive so restore inputs can be read and publish outputs can be written.

In [ ]:
from google.colab import drive


drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print(f"Drive mounted: {DRIVE_PROJECT_ROOT.parent}")

## 3.2 System packages

Ensure the runtime has the system tools required by workflow-provided archive commands.

In [ ]:
import shutil


if shutil.which("zstd") is None:
    !sudo apt-get update
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")

    !sudo apt-get install -y zstd
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
    print("Installed zstd.")
else:
    print("zstd is already available.")

!zstd --version
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("zstd command is not available after preflight.")


## 3.3 Repository checkout

Clone or update the requested repository revision.

In [ ]:
%cd /content

if PROJECT_ROOT.exists():
    print(f"Removing stale repository checkout: {PROJECT_ROOT}")
    !rm -rf {PROJECT_ROOT}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(f"Failed to remove existing directory: {PROJECT_ROOT}")

!git clone {REPO_URL} {PROJECT_ROOT}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to clone repository.")

!git -C {PROJECT_ROOT} checkout {REPO_REF}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF}.")

print(f"Repository ready: {PROJECT_ROOT}")
!git -C {PROJECT_ROOT} rev-parse HEAD
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to determine checked out revision.")

## 3.4 Install dependencies

Install the repository package and notebook runtime dependencies.

In [ ]:
%cd {PROJECT_ROOT}
%pip install --upgrade pip
%pip install -r "requirements-colab.txt"
print("Repository dependencies installed from requirements-colab.txt.")

## 3.5 Add source tree to import path

Prefer the checked-out source tree for workflow imports in this runtime.

In [ ]:
import sys


%cd {PROJECT_ROOT}
source_path = PROJECT_ROOT / "src"
if str(source_path) not in sys.path:
    sys.path.append(str(source_path))
print(f"Repository src directory available on sys.path: {source_path}")

# 4. Notebook Helpers

Define small notebook-local helpers for visible execution and readable summaries. These helpers do not build workflow plans, infer path ownership, validate business rules, or reach into non-public workflow internals.

## 4.1 Execution helpers

Run workflow-provided shell commands.

In [ ]:
def run_command_spec(spec):
    command = spec.command
    print(f"[{command.label}]")
    print(command.bash)
    !bash -o pipefail -c {command.arg}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(command.failure)


def run_command_specs(specs):
    for spec in specs:
        run_command_spec(spec)

## 4.2 Display helpers

Print compact generic summaries without changing workflow state.

In [ ]:
def _render_text(value) -> str:
    if value is None:
        return "none"
    return str(value)


def _render_key(value) -> str:
    return _render_text(getattr(value, "value", value))


def display_block_title(title, *, leading_blank: bool = False) -> None:
    rendered_title = str(title).strip()
    if not rendered_title:
        rendered_title = "Untitled"

    if leading_blank:
        print()

    print(rendered_title)
    print("-" * len(rendered_title))


def display_lines(title, lines, *, leading_blank: bool = False) -> None:
    rendered_lines = tuple(lines)

    display_block_title(title, leading_blank=leading_blank)

    if not rendered_lines:
        print("- none")
        return

    for line in rendered_lines:
        print(f"- {_render_text(line)}")


def display_nested_counts(title, counts, *, leading_blank: bool = False) -> None:
    display_block_title(title, leading_blank=leading_blank)

    if not counts:
        print("- none")
        return

    for tier, split_counts in counts.items():
        rendered_counts = ", ".join(
            f"{_render_key(split)}: {count}"
            for split, count in split_counts.items()
        )
        print(f"- {_render_key(tier)}: {rendered_counts}")


def display_review_sections(sections, *, leading_blank: bool = False) -> None:
    rendered_sections = tuple(sections)

    if not rendered_sections:
        display_block_title("Review", leading_blank=leading_blank)
        print("- none")
        return

    for section_index, section in enumerate(rendered_sections):
        display_block_title(
            section.title,
            leading_blank=leading_blank or section_index > 0,
        )

        items = tuple(section.items)
        if not items:
            print("- none")
            continue

        for item_index, item in enumerate(items, start=1):
            if item_index > 1:
                print()

            print(f"{item_index}. {_render_text(item.label)}")

            fields = tuple(item.fields)
            if not fields:
                print("   - none")
                continue

            for field in fields:
                print(f"   {field.label}: {_render_text(field.value)}")

# 5. Workflow Setup

Import the public `workflows.tiers` API, build the tiers workflow config, and ask the workflow to build the restore runtime plan. The outputs of this section are `tiers_config` and `tiers_runtime_plan`.

## 5.1 Import public tiers API

Use only the notebook-facing public workflow surface.

In [ ]:
from text_to_sign_production.workflows.tiers import (
    TiersWorkflowConfig,
    build_tiers_publish_plan,
    build_tiers_runtime_plan,
    run_tiers_workflow,
    summarize_tiers_publish_plan,
    summarize_tiers_publish_plan_review,
    summarize_tiers_runtime_plan_review,
    summarize_tiers_workflow_outputs,
    validate_tiers_inputs,
    verify_tiers_runtime_inputs,
)

## 5.2 Build config and runtime plan

Construct the workflow config from operator settings and review the restore plan counts.

In [ ]:
tiers_config = TiersWorkflowConfig(
    project_root=PROJECT_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
    splits=TIERS_SPLITS,
    filters_config_relative_path=FILTERS_CONFIG_RELATIVE_PATH,
    tiers_config_relative_path=TIERS_CONFIG_RELATIVE_PATH,
)
tiers_runtime_plan = build_tiers_runtime_plan(tiers_config)

print(f"Project root: {tiers_runtime_plan.project_root}")
print(f"Drive project root: {tiers_runtime_plan.drive_project_root}")
print(f"Splits: {tuple(split.value for split in tiers_config.splits)}")
print(f"Restore file transfers: {len(tiers_runtime_plan.restore_file_transfers)}")
print(f"Restore archive extracts: {len(tiers_runtime_plan.restore_archive_extracts)}")

# 6. Restore Inputs

Review and restore the runtime inputs required by the tiers workflow.

## 6.1 Review restore plan

Review untiered passed manifest restore specs and passed sample archive extract specs. This is a review-only checkpoint; it does not execute restore work.

In [ ]:
untiered_passed_manifest_restore_specs = tiers_runtime_plan.restore_file_transfers
passed_sample_archive_extract_specs = tiers_runtime_plan.restore_archive_extracts

tiers_restore_review_sections = summarize_tiers_runtime_plan_review(tiers_runtime_plan)
display_review_sections(tiers_restore_review_sections)

## 6.2 Restore inputs and verify readiness

Operator action: execute the reviewed restore specs, then verify runtime inputs and validate workflow inputs in sequence.

In [ ]:
print("Restore plan reviewed.")
print("Restoring untiered passed manifests...")
run_command_specs(untiered_passed_manifest_restore_specs)
print("Untiered passed manifests restored.")

print("Extracting passed sample archives...")
run_command_specs(passed_sample_archive_extract_specs)
print("Passed sample archive extraction complete.")

print("Verifying runtime inputs...")
verify_tiers_runtime_inputs(tiers_config)
print("Runtime inputs verified.")

print("Validating workflow inputs...")
validate_tiers_inputs(tiers_config)
print("Workflow inputs validated.")

print("Restore complete.")

# 7. Run Tiers Workflow

Run the public tiers workflow and review the workflow output summary.

## 7.1 Run workflow

Operator action: run the tiers workflow for the configured splits.

In [ ]:
print("Running tiers workflow...")
tiers_result = run_tiers_workflow(tiers_config)
print("Tiers workflow complete.")

## 7.2 Review workflow outputs

Review included/excluded counts, tiered manifests, and reports from the workflow result.

In [ ]:
tiers_output_summary = summarize_tiers_workflow_outputs(tiers_result)

display_nested_counts(
    "Included counts by tier by split",
    tiers_output_summary.included_counts_by_tier_by_split,
)
display_nested_counts(
    "Excluded counts by tier by split",
    tiers_output_summary.excluded_counts_by_tier_by_split,
    leading_blank=True,
)
display_lines(
    "Tiered manifest output paths",
    tiers_output_summary.tiered_manifest_paths,
    leading_blank=True,
)
display_lines("Report paths", tiers_output_summary.report_paths, leading_blank=True)

# 8. Publish Outputs

Build, review, and execute the file-only publish plan for tiers workflow outputs.

## 8.1 Build and review publish plan

Review the planned publish targets and file publish specs before execution. This is a publish-plan review surface and plan-summary data, not an execution-result summary unless the public API truly provides one.

In [ ]:
print("Building publish plan...")
tiers_publish_plan = build_tiers_publish_plan(tiers_config, tiers_result)
tiers_publish_plan_summary = summarize_tiers_publish_plan(tiers_publish_plan)

publish_file_transfer_specs = tiers_publish_plan.file_transfers
tiers_publish_review_sections = summarize_tiers_publish_plan_review(tiers_publish_plan)

print("Publish plan ready.")
display_block_title("Publish plan summary", leading_blank=True)
print(f"- planned files: {tiers_publish_plan_summary.planned_file_count}")
display_lines(
    "Publish target paths",
    tiers_publish_plan_summary.target_paths,
    leading_blank=True,
)
display_review_sections(tiers_publish_review_sections, leading_blank=True)

## 8.2 Execute publish plan

Operator action: publish tiered manifests and tiers reports using the reviewed file publish specs.

In [ ]:
print("Publishing tiered manifests and reports...")
run_command_specs(publish_file_transfer_specs)
print("File publish complete.")

print("Publish action complete.")

# 9. Final Summary

Final operator summary for the configured tiers run. Publish summary reflects planned outputs only; the current public API does not expose a publish execution-result DTO.

In [ ]:
display_block_title("Final operator summary")
print(f"Requested splits: {tuple(split.value for split in tiers_config.splits)}")
display_nested_counts(
    "Included counts by tier by split",
    tiers_output_summary.included_counts_by_tier_by_split,
    leading_blank=True,
)
display_nested_counts(
    "Excluded counts by tier by split",
    tiers_output_summary.excluded_counts_by_tier_by_split,
    leading_blank=True,
)
display_lines(
    "Tiered manifest output paths",
    tiers_output_summary.tiered_manifest_paths,
    leading_blank=True,
)
display_lines("Report paths", tiers_output_summary.report_paths, leading_blank=True)
display_lines(
    "Publish target paths",
    tiers_publish_plan_summary.target_paths,
    leading_blank=True,
)
print("Publish summary reflects planned outputs only.")
print("The current public API does not expose a publish execution-result DTO.")